# 01. Initialize Phoenix Tracing

## Why are we implementing this?

Phoenix provides observability for our SEVA_AI retrieval pipeline.

We use OpenTelemetry and OpenInference to capture execution traces from LangChain components.

This allows us to inspect:
- Retrieval execution
- Retrieved documents
- Execution latency
- Nested operations
- Errors and failures

We will send traces to our locally running Phoenix server at http://localhost:6006.

Our Phoenix project name will be SEVA_AI.

In [1]:
import phoenix.otel
import openinference.instrumentation.langchain

print("Phoenix instrumentation installed successfully!")

Phoenix instrumentation installed successfully!


In [2]:
from phoenix.otel import register
from opentelemetry import trace

tracer_provider = register(
    project_name="SEVA_AI",
    endpoint="http://localhost:6006/v1/traces",
    protocol="http/protobuf",
    batch=True,
    auto_instrument=True
)

tracer = trace.get_tracer("seva_ai.test")

with tracer.start_as_current_span("phoenix_connection_test") as span:
    span.set_attribute("project", "SEVA_AI")
    span.set_attribute("operation", "connection_test")
    span.set_attribute("status", "success")

tracer_provider.force_flush()

print("Phoenix trace submitted.")

OpenTelemetry Tracing Details
|  Phoenix Project: SEVA_AI
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix trace submitted.


## 02. Create First Test Trace

### Why are we implementing this?

We want to verify end-to-end communication between our Python application and Phoenix.

We will manually create a test span using the OpenTelemetry API and export it to our SEVA_AI Phoenix project.

This confirms that our tracing configuration works before instrumenting the actual retrieval pipeline.

In [3]:
from opentelemetry import trace

tracer = trace.get_tracer("seva_ai.test")

with tracer.start_as_current_span("phoenix_connection_test") as span:
    span.set_attribute("project", "SEVA_AI")
    span.set_attribute("operation", "connection_test")
    span.set_attribute("status", "success")

print("Test trace exported to Phoenix.")

Failed to export spans batch code: 405, reason: Method Not Allowed


Test trace exported to Phoenix.


## 03. Chroma Discovery Retrieval Tracing

### Why are we implementing this?

We want to observe the actual SEVA_AI discovery retrieval operation inside Phoenix.

We will capture:

- User query
- Retrieved chunks
- Scheme IDs
- Similarity distances
- Number of retrieved chunks
- Retrieval execution time
- Retrieval errors

This will help us investigate why certain schemes are retrieved or missed.

We will preserve our existing Chroma collection and retrieval configuration.

In [3]:
import json
import time

from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode

tracer = trace.get_tracer("seva_ai.discovery")


def traced_discovery_retrieval(query: str, chunk_k: int = 30):

    with tracer.start_as_current_span(
        "seva_discovery_retrieval"
    ) as span:

        span.set_attribute(
            "openinference.span.kind", "RETRIEVER"
        )
        span.set_attribute("input.value", query)
        span.set_attribute("retrieval.top_k", chunk_k)

        start_time = time.perf_counter()

        try:
            results = discovery_db.similarity_search_with_score(
                query,
                k=chunk_k
            )

            latency_ms = (
                time.perf_counter() - start_time
            ) * 1000

            retrieved_documents = []

            for doc, distance in results:
                retrieved_documents.append({
                    "id": str(doc.metadata.get("parent_id", "")),
                    "content": doc.page_content[:1000],
                    "score": float(distance),
                    "metadata": {
                        "scheme_id": str(
                            doc.metadata.get("scheme_id", "")
                        ),
                        "source_field": str(
                            doc.metadata.get("source_field", "")
                        )
                    }
                })

            span.set_attribute(
                "retrieval.documents",
                json.dumps(retrieved_documents, ensure_ascii=False)
            )

            span.set_attribute(
                "output.value",
                json.dumps({
                    "retrieved_chunks": len(results),
                    "latency_ms": round(latency_ms, 2)
                })
            )

            span.set_attribute("retrieval.latency_ms", latency_ms)

            return results

        except Exception as error:
            span.record_exception(error)
            span.set_status(
                Status(StatusCode.ERROR, str(error))
            )
            raise


print("Discovery retrieval tracing function initialized.")

Discovery retrieval tracing function initialized.


## 04. Execute Discovery Retrieval

### Why are we implementing this?

We need to execute a real discovery query to generate a Phoenix trace.

We will use one of our existing scheme-aware FAQ queries and retrieve the top 30 chunks.

The retrieved results will be visible in the Phoenix trace details.

## 04. Prepare Scheme-Aware Test Query

### Why are we implementing this?

The golden test JSON contains the original scheme name and FAQ question.

We need to reconstruct the scheme-aware retrieval query because the retrieval_query column was created dynamically in our previous notebook and was not necessarily saved to the JSON file.

We will preserve the original golden dataset.

In [6]:
import json
from pathlib import Path

gd = Path(
    r"G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\final\Final_dataset_splits\test.json"
)

with open(gd, "r", encoding="utf-8") as f:
    golden_df = json.load(f)

test_query = (
    f"For the scheme '{golden_df[0]['scheme_name']}': "
    f"{golden_df[0]['question']}"
)

print("Scheme:", golden_df[0]["scheme_name"])
print("Original question:", golden_df[0]["question"])
print("Retrieval query:", test_query)

Scheme: Dayanand Bandodkar Scheme For Higher Education For Orphans
Original question: Is this scheme limited to certain fields of study or open to all disciplines?
Retrieval query: For the scheme 'Dayanand Bandodkar Scheme For Higher Education For Orphans': Is this scheme limited to certain fields of study or open to all disciplines?


## 05. Execute Discovery Retrieval

### Why are we implementing this?

We want to execute a real Chroma discovery search and send its execution trace to Phoenix.

We will retrieve the top 30 chunks using our scheme-aware query.

In [1]:
import json
import time
from pathlib import Path
from collections import defaultdict

from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

In [2]:
OUTPUT_DIR1 = Path("G:\\SEVA_Ai\\SEVA_Ai\\output\\SEVA_CHILDREN")

VECTORSTORE_DIR = OUTPUT_DIR1 / "vectorstores"

PARENT_PATH = OUTPUT_DIR1 / "parent_documents.jsonl"

print("Parent file exists:", PARENT_PATH.exists())
print("Vectorstore directory exists:", VECTORSTORE_DIR.exists())

Parent file exists: True
Vectorstore directory exists: True


## 01. Download Embedding Model Locally

### Why are we implementing this?

Our project uses sentence-transformers/all-MiniLM-L6-v2.

Currently, model initialization may repeatedly involve Hugging Face cache checks or downloading model files.

We want to maintain one local copy of the model inside our project.

Benefits:
- Avoid repeated model downloads.
- Work without internet after downloading.
- Reuse the same model files across notebooks.
- Maintain consistent embeddings across retrieval experiments.

In [3]:
import os

# Must be configured before importing huggingface_hub
os.environ["HF_HUB_DISABLE_XET"] = "1"

from huggingface_hub import snapshot_download
from pathlib import Path

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

MODEL_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2"
)

MODEL_PATH.mkdir(parents=True, exist_ok=True)

download_path = snapshot_download(
    repo_id=MODEL_NAME,
    local_dir=str(MODEL_PATH),
    max_workers=2
)

print("Model downloaded successfully!")
print("Location:", download_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Model downloaded successfully!
Location: G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2


## 01. Verify Local Embedding Model

### Why are we implementing this?

We have downloaded all-MiniLM-L6-v2 from Hugging Face.

Before loading it, we need to verify that the model weights, tokenizer and configuration files exist in our local directory.

This ensures that we can load the model without downloading files again.

In [5]:
from pathlib import Path

MODEL_PATH = Path(
    r"G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2"
)

required_files = [
    "config.json",
    "model.safetensors",
    "modules.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.txt",
    "1_Pooling/config.json"
]

missing_files = [
    file for file in required_files
    if not (MODEL_PATH / file).exists()
]

if not missing_files:
    print("Model files verified successfully!")
    print("Model location:", MODEL_PATH)
else:
    print("Missing files:", missing_files)

Model files verified successfully!
Model location: G:\SEVA_Ai\SEVA_Ai\models\all-MiniLM-L6-v2


In [6]:
import os
import time

os.environ["HF_HUB_OFFLINE"] = "1"

from langchain_huggingface import HuggingFaceEmbeddings

start_time = time.perf_counter()

embedding_model = HuggingFaceEmbeddings(
    model_name=str(MODEL_PATH),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

load_time = time.perf_counter() - start_time

test_embedding = embedding_model.embed_query(
    "What government schemes are available for farmers?"
)

print(f"Model loading time: {load_time:.2f} seconds")
print("Embedding dimensions:", len(test_embedding))
print("Model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loading time: 192.34 seconds
Embedding dimensions: 384
Model loaded successfully!


In [7]:
discovery_db = Chroma(
    collection_name="seva_discovery_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "discovery")
)

detail_db = Chroma(
    collection_name="seva_detail_v1",
    embedding_function=embedding_model,
    persist_directory=str(VECTORSTORE_DIR / "detail")
)

discovery_count = discovery_db._collection.count()
detail_count = detail_db._collection.count()

print("Discovery collection:", discovery_count)
print("Detail collection:", detail_count)
print("Total indexed chunks:", discovery_count + detail_count)

Discovery collection: 23438
Detail collection: 67945
Total indexed chunks: 91383


In [8]:
discovery_results = traced_discovery_retrieval(
    query=test_query,
    chunk_k=30
)

tracer_provider.force_flush()

print("Retrieved chunks:", len(discovery_results))
print("Phoenix trace exported.")

NameError: name 'traced_discovery_retrieval' is not defined